In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Preview the bottom-32 and top-32 MNIST samples by log-volume (train & held-out).

Assumes the NPZ has keys:
  - logv_train, logv_val          (float arrays aligned to train_indices/val_indices)
  - train_indices, val_indices    (indices into MNIST(train=True))

Outputs:
  members_preview_lowest32.png,   heldout_preview_lowest32.png
  members_preview_highest32.png,  heldout_preview_highest32.png
"""
import os
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import MNIST
from torchvision import transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt

# ----------------------------
# config
# ----------------------------
dataset_root = "/home/ethanrao/MIA_LDM/data"            # <- change this
logvol_npz   = "/home/ethanrao/MIA_LDM/data/logvols_mnist_beta_1e_2.npz"     # <- change this
img_size     = 32
batch_size   = 32
num_workers  = 4
k_extreme    = 32
save_dir     = "/banana/ethan/MIA_LDM_data/MNIST_KL_sweep/1e_2/mnist_extreme_previews"
os.makedirs(save_dir, exist_ok=True)

# ----------------------------
# transforms & helpers
# ----------------------------
tf = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

def _denorm(imgs: torch.Tensor) -> torch.Tensor:
    # imgs: (B,1,H,W) normalized to [-1,1] -> back to [0,1]
    return torch.clamp(imgs * 0.5 + 0.5, 0.0, 1.0)

def preview_batch(loader, title, nrow=8, save_path=None):
    imgs, labs = next(iter(loader))
    print(f"[{title}] imgs={tuple(imgs.shape)}, dtype={imgs.dtype}; "
          f"labels shape={tuple(labs.shape)}; min={imgs.min().item():.3f}, max={imgs.max().item():.3f}")

    # label histogram
    uniq, cnts = np.unique(labs.numpy(), return_counts=True)
    hist_str = ", ".join([f"{int(u)}:{int(c)}" for u, c in zip(uniq, cnts)])
    print(f"[{title}] digit counts: {hist_str}")

    imgs_dn = _denorm(imgs)   # (B,1,H,W)
    grid = make_grid(imgs_dn, nrow=nrow, padding=2)  # (1,H,W) -> grid (1,H,W)
    grid_np = grid.permute(1, 2, 0).squeeze(-1).numpy()  # HxW for imshow (grayscale)

    rows = (imgs.size(0) + nrow - 1) // nrow
    plt.figure(figsize=(nrow * 1.6, rows * 1.6))
    plt.imshow(grid_np, cmap="gray")
    plt.title(f"{title} — batch_size={imgs.size(0)}")
    plt.axis('off')

    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
        print(f"[{title}] saved preview to: {save_path}")
        plt.close()
    else:
        plt.show()

# ----------------------------
# load MNIST (train=True holds both "members" and "heldout" subsets by index)
# ----------------------------
full_train = MNIST(root=dataset_root, train=True, download=True, transform=tf)

# ----------------------------
# load log-volume npz and pick extremes
# ----------------------------
data = np.load(logvol_npz)
required = ["logv_train", "logv_val", "train_indices", "val_indices"]
missing = [k for k in required if k not in data]
if missing:
    raise KeyError(f"NPZ missing keys: {missing}. Found keys: {list(data.keys())}")

logv_train = data["logv_train"].astype(np.float64)
logv_val   = data["logv_val"].astype(np.float64)
train_idx  = data["train_indices"].astype(np.int64)
val_idx    = data["val_indices"].astype(np.int64)

# safety: clamp k to available sizes
k = int(min(k_extreme, len(train_idx), len(val_idx)))
if k < k_extreme:
    print(f"[warn] Not enough samples to pick {k_extreme}; using k={k}")

# argsort gives ascending -> first k = lowest, last k = highest
tr_sorted = np.argsort(logv_train)
va_sorted = np.argsort(logv_val)

tr_low_indices  = train_idx[tr_sorted[:k]]
tr_high_indices = train_idx[tr_sorted[-k:]]
va_low_indices  = val_idx[va_sorted[:k]]
va_high_indices = val_idx[va_sorted[-k:]]

print(f"[extreme] selected lowest {k} and highest {k} (train & held-out)")

# ----------------------------
# build DataLoaders for each extreme bucket (batch=32 to fit exactly one grid)
# ----------------------------
m_low_loader  = DataLoader(Subset(full_train, tr_low_indices.tolist()),
                           batch_size=batch_size, shuffle=False,
                           num_workers=num_workers, pin_memory=True)
m_high_loader = DataLoader(Subset(full_train, tr_high_indices.tolist()),
                           batch_size=batch_size, shuffle=False,
                           num_workers=num_workers, pin_memory=True)
h_low_loader  = DataLoader(Subset(full_train, va_low_indices.tolist()),
                           batch_size=batch_size, shuffle=False,
                           num_workers=num_workers, pin_memory=True)
h_high_loader = DataLoader(Subset(full_train, va_high_indices.tolist()),
                           batch_size=batch_size, shuffle=False,
                           num_workers=num_workers, pin_memory=True)

# ----------------------------
# preview and save grids
# ----------------------------
preview_batch(m_low_loader,  "members/train lowest",  nrow=8,
              save_path=os.path.join(save_dir, "members_preview_lowest32.png"))
preview_batch(m_high_loader, "members/train highest", nrow=8,
              save_path=os.path.join(save_dir, "members_preview_highest32.png"))
preview_batch(h_low_loader,  "heldout/val lowest",    nrow=8,
              save_path=os.path.join(save_dir, "heldout_preview_lowest32.png"))
preview_batch(h_high_loader, "heldout/val highest",   nrow=8,
              save_path=os.path.join(save_dir, "heldout_preview_highest32.png"))


[extreme] selected lowest 32 and highest 32 (train & held-out)
[members/train lowest] imgs=(32, 1, 32, 32), dtype=torch.float32; labels shape=(32,); min=-1.000, max=0.992
[members/train lowest] digit counts: 1:32
[members/train lowest] saved preview to: /banana/ethan/MIA_LDM_data/MNIST_KL_sweep/1e_2/mnist_extreme_previews/members_preview_lowest32.png
[members/train highest] imgs=(32, 1, 32, 32), dtype=torch.float32; labels shape=(32,); min=-1.000, max=1.000
[members/train highest] digit counts: 0:4, 2:4, 3:5, 5:3, 6:1, 8:13, 9:2
[members/train highest] saved preview to: /banana/ethan/MIA_LDM_data/MNIST_KL_sweep/1e_2/mnist_extreme_previews/members_preview_highest32.png
[heldout/val lowest] imgs=(32, 1, 32, 32), dtype=torch.float32; labels shape=(32,); min=-1.000, max=1.000
[heldout/val lowest] digit counts: 1:32
[heldout/val lowest] saved preview to: /banana/ethan/MIA_LDM_data/MNIST_KL_sweep/1e_2/mnist_extreme_previews/heldout_preview_lowest32.png
[heldout/val highest] imgs=(32, 1, 32, 

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Preview the bottom-32 and top-32 MNIST samples by log-volume (train & held-out).

Assumes the NPZ has keys:
  - logv_train, logv_val          (float arrays aligned to train_indices/val_indices)
  - train_indices, val_indices    (indices into MNIST(train=True))

Outputs:
  members_preview_lowest32.png,   heldout_preview_lowest32.png
  members_preview_highest32.png,  heldout_preview_highest32.png
"""
import os
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import MNIST, CIFAR10
from torchvision import transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt

# ----------------------------
# config
# ----------------------------
dataset_root = "/home/ethanrao/MIA_LDM/data"            # <- change this
logvol_npz   = "/home/ethanrao/MIA_LDM/data/logvols_cifar10_beta_1e_2.npz"     # <- change this
img_size     = 32
batch_size   = 32
num_workers  = 4
k_extreme    = 32
save_dir     = "/banana/ethan/MIA_LDM_data/KL_sweep/1e_2/cifar10_extreme_previews"
os.makedirs(save_dir, exist_ok=True)

# ----------------------------
# transforms & helpers
# ----------------------------
tf = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

def _denorm(imgs: torch.Tensor) -> torch.Tensor:
    # imgs: (B,1,H,W) normalized to [-1,1] -> back to [0,1]
    return torch.clamp(imgs * 0.5 + 0.5, 0.0, 1.0)

def preview_batch(loader, title, nrow=8, save_path=None):
    imgs, labs = next(iter(loader))
    print(f"[{title}] imgs={tuple(imgs.shape)}, dtype={imgs.dtype}; "
          f"labels shape={tuple(labs.shape)}; min={imgs.min().item():.3f}, max={imgs.max().item():.3f}")

    # label histogram
    uniq, cnts = np.unique(labs.numpy(), return_counts=True)
    hist_str = ", ".join([f"{int(u)}:{int(c)}" for u, c in zip(uniq, cnts)])
    print(f"[{title}] digit counts: {hist_str}")

    imgs_dn = _denorm(imgs)   # (B,1,H,W)
    grid = make_grid(imgs_dn, nrow=nrow, padding=2)  # (1,H,W) -> grid (1,H,W)
    grid_np = grid.permute(1, 2, 0).squeeze(-1).numpy()  # HxW for imshow (grayscale)

    rows = (imgs.size(0) + nrow - 1) // nrow
    plt.figure(figsize=(nrow * 1.6, rows * 1.6))
    plt.imshow(grid_np, cmap="gray")
    plt.title(f"{title} — batch_size={imgs.size(0)}")
    plt.axis('off')

    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
        print(f"[{title}] saved preview to: {save_path}")
        plt.close()
    else:
        plt.show()

# ----------------------------
# load MNIST (train=True holds both "members" and "heldout" subsets by index)
# ----------------------------
full_train = CIFAR10(root=dataset_root, train=True, download=True, transform=tf)

# ----------------------------
# load log-volume npz and pick extremes
# ----------------------------
data = np.load(logvol_npz)
required = ["logv_train", "logv_val", "train_indices", "val_indices"]
missing = [k for k in required if k not in data]
if missing:
    raise KeyError(f"NPZ missing keys: {missing}. Found keys: {list(data.keys())}")

logv_train = data["logv_train"].astype(np.float64)
logv_val   = data["logv_val"].astype(np.float64)
train_idx  = data["train_indices"].astype(np.int64)
val_idx    = data["val_indices"].astype(np.int64)

# safety: clamp k to available sizes
k = int(min(k_extreme, len(train_idx), len(val_idx)))
if k < k_extreme:
    print(f"[warn] Not enough samples to pick {k_extreme}; using k={k}")

# argsort gives ascending -> first k = lowest, last k = highest
tr_sorted = np.argsort(logv_train)
va_sorted = np.argsort(logv_val)

tr_low_indices  = train_idx[tr_sorted[:k]]
tr_high_indices = train_idx[tr_sorted[-k:]]
va_low_indices  = val_idx[va_sorted[:k]]
va_high_indices = val_idx[va_sorted[-k:]]

print(f"[extreme] selected lowest {k} and highest {k} (train & held-out)")

# ----------------------------
# build DataLoaders for each extreme bucket (batch=32 to fit exactly one grid)
# ----------------------------
m_low_loader  = DataLoader(Subset(full_train, tr_low_indices.tolist()),
                           batch_size=batch_size, shuffle=False,
                           num_workers=num_workers, pin_memory=True)
m_high_loader = DataLoader(Subset(full_train, tr_high_indices.tolist()),
                           batch_size=batch_size, shuffle=False,
                           num_workers=num_workers, pin_memory=True)
h_low_loader  = DataLoader(Subset(full_train, va_low_indices.tolist()),
                           batch_size=batch_size, shuffle=False,
                           num_workers=num_workers, pin_memory=True)
h_high_loader = DataLoader(Subset(full_train, va_high_indices.tolist()),
                           batch_size=batch_size, shuffle=False,
                           num_workers=num_workers, pin_memory=True)

# ----------------------------
# preview and save grids
# ----------------------------
preview_batch(m_low_loader,  "members/train lowest",  nrow=8,
              save_path=os.path.join(save_dir, "members_preview_lowest32.png"))
preview_batch(m_high_loader, "members/train highest", nrow=8,
              save_path=os.path.join(save_dir, "members_preview_highest32.png"))
preview_batch(h_low_loader,  "heldout/val lowest",    nrow=8,
              save_path=os.path.join(save_dir, "heldout_preview_lowest32.png"))
preview_batch(h_high_loader, "heldout/val highest",   nrow=8,
              save_path=os.path.join(save_dir, "heldout_preview_highest32.png"))


Files already downloaded and verified
[extreme] selected lowest 32 and highest 32 (train & held-out)
[members/train lowest] imgs=(32, 3, 32, 32), dtype=torch.float32; labels shape=(32,); min=-1.000, max=1.000
[members/train lowest] digit counts: 0:19, 2:8, 4:1, 6:1, 8:3
[members/train lowest] saved preview to: /banana/ethan/MIA_LDM_data/KL_sweep/1e_2/cifar10_extreme_previews/members_preview_lowest32.png
[members/train highest] imgs=(32, 3, 32, 32), dtype=torch.float32; labels shape=(32,); min=-1.000, max=1.000
[members/train highest] digit counts: 0:1, 1:4, 2:3, 3:1, 4:1, 5:6, 6:1, 7:4, 8:1, 9:10
[members/train highest] saved preview to: /banana/ethan/MIA_LDM_data/KL_sweep/1e_2/cifar10_extreme_previews/members_preview_highest32.png
[heldout/val lowest] imgs=(32, 3, 32, 32), dtype=torch.float32; labels shape=(32,); min=-1.000, max=1.000
[heldout/val lowest] digit counts: 0:22, 2:4, 3:1, 4:2, 6:1, 8:2
[heldout/val lowest] saved preview to: /banana/ethan/MIA_LDM_data/KL_sweep/1e_2/cifar10